In [30]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold

In [31]:
train = pd.read_csv("train.csv")
y = np.log1p(train["hours_spent"])
X = train.drop(columns=["hours_spent", "log_hours", "hours_bucket", "project_id"])

def add_features(df):
    df = df.copy()
    df["biggest_commit_share"] = df["max_lines_in_one_commit"] / df["lines_added_total"].replace(0, np.nan)
    df["lines_per_active_day"] = df["lines_of_code"] / df["active_commit_days"].replace(0, np.nan)
    df["commits_per_active_day"] = df["num_commits"] / df["active_commit_days"].replace(0, np.nan)
    df["active_day_ratio"] = df["active_commit_days"] / (df["project_span_days"] + 1)
    return df

In [32]:
ORDER_MAPS = {
    "author_prior_projects":     {"0": 0, "1-2": 1, "3-5": 2, "6+": 3},
    "author_account_age_years":  {"<1": 0, "1-2": 1, "3-5": 2, "6+": 3},
}
def encode_ordered(df):
    df = df.copy()
    for col, mapping in ORDER_MAPS.items():
        df[col] = df[col].astype(str).map(mapping)
    return df

In [33]:
def encode_categories(df):
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype("category")
    return df

In [34]:
def encode_yes_no(df):
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        values = set(df[col].dropna().astype(str).unique())
        if values <= {"True", "False"}:
            df[col] = df[col].astype(str).map({"True": 1.0, "False": 0.0})
    return df

In [35]:
def prepare(df):
    df = add_features(df)
    df = encode_ordered(df)
    df = encode_yes_no(df)
    df = encode_categories(df)
    return df

X = prepare(X)

C:\Users\Fujitsu\AppData\Local\Temp\ipykernel_9320\1303781909.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\Fujitsu\AppData\Local\Temp\ipykernel_9320\3138021966.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/

In [36]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(X.shape)

(4500, 73)


In [37]:
from sklearn.metrics import root_mean_squared_log_error

def cv_score(make_model, X_data):
    oof = np.zeros(len(X_data))
    for train_idx, test_idx in kf.split(X_data):
        model = make_model()
        model.fit(X_data.iloc[train_idx], y.iloc[train_idx])
        oof[test_idx] = model.predict(X_data.iloc[test_idx])
    pred = np.expm1(oof).clip(0, 140.7)
    return root_mean_squared_log_error(train["hours_spent"], pred)

In [38]:
bool_cols = X.select_dtypes(include="bool").columns
X = X.astype({c: int for c in bool_cols})


cat_cols = X.select_dtypes(include="category").columns.tolist()
X_cat = X.copy()
for c in cat_cols:
    X_cat[c] = X_cat[c].astype(object).fillna("missing").astype(str)

In [ ]:
import xgboost as xgb
from catboost import CatBoostRegressor

make_lgb = lambda: lgb.LGBMRegressor(
    n_estimators=800, learning_rate=0.03, num_leaves=31, min_child_samples=20,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=42, verbose=-1)

make_xgb = lambda: xgb.XGBRegressor(
    n_estimators=800, learning_rate=0.03, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    enable_categorical=True, tree_method="hist", random_state=42)

make_cat = lambda: CatBoostRegressor(
    iterations=1500, learning_rate=0.03, depth=6,
    cat_features=cat_cols, random_seed=42, verbose=0)

print("LightGBM:", round(cv_score(make_lgb, X), 4))
print("XGBoost: ", round(cv_score(make_xgb, X), 4))
print("CatBoost:", round(cv_score(make_cat, X_cat), 4))

LightGBM: 0.6643


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

d:\Xin Bing\VSC\hour-detective\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def objective(trial):
    params = dict(
        n_estimators = trial.suggest_int("n_estimators", 300, 2000),
        learning_rate = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves = trial.suggest_int("num_leaves", 7, 63),
        min_child_samples = trial.suggest_int("min_child_samples", 5, 100),
        subsample = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.3, 1.0),
        reg_lambda = trial.suggest_float("reg_lambda", 0.001, 10, log=True),
    )
    make = lambda: lgb.LGBMRegressor(**params, subsample_freq=1, random_state=42, verbose=-1)
    return cv_score(make, X)

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best score: ", round(study.best_value, 4))
print("Best settings: ", study.best_params)

Best trial: 39. Best value: 0.655048: 100%|██████████| 50/50 [04:39<00:00,  5.60s/it]

Best score:  0.655
Best settings:  {'n_estimators': 801, 'learning_rate': 0.01010539335780274, 'num_leaves': 35, 'min_child_samples': 50, 'subsample': 0.7520377986379265, 'colsample_bytree': 0.5994064834742668, 'reg_lambda': 0.35690477098849843}


In [ ]:
def cv_oof(make_model, X_data):
    oof = np.zeros(len(X_data))
    for train_idx, test_idx in kf.split(X_data):
        model = make_model()
        model.fit(X_data.iloc[train_idx], y.iloc[train_idx])
        oof[test_idx] = model.predict(X_data.iloc[test_idx])
    return oof

In [ ]:
def score_from_oof(oof):
    pred = np.expm1(oof).clip(0, 140.7)
    return root_mean_squared_log_error(train["hours_spent"], pred)

In [ ]:
best_params = study.best_params
make_lgb_best = lambda: lgb.LGBMRegressor(**best_params, subsample_freq=1, random_state=42, verbose=-1)

oof_lgb = cv_oof(make_lgb, X)
oof_xgb = cv_oof(make_xgb, X)
oof_cat = cv_oof(make_cat, X_cat)

print("LightGBM: ", round(score_from_oof(oof_lgb), 4))
print("XGBoost: ", round(score_from_oof(oof_xgb), 4))
print("CatBoost: ", round(score_from_oof(oof_cat), 4))

blend = (oof_lgb + oof_xgb + oof_cat) / 3
print("Average of all 3:", round(score_from_oof(blend), 4))

KeyboardInterrupt: 